# AgentReach AI — Phase 2: Google Drive (read-only)

This notebook extends the tested Needle 3 prototype with a **read-only Google Drive search tool** while retaining public GitHub lookup.

## Boundaries
- Drive tool searches filenames and returns limited metadata only: file ID, name, type, last-modified time, and a link.
- No file contents are read in this phase. No files are created, edited, moved, shared, or deleted.
- Google authorization is performed by you in Colab. Review the authorization prompt and continue only if you trust the notebook code and are comfortable granting the requested access.
- The code below exposes only the listed read-only actions to Needle. Google authorization may grant the notebook broader account access than those individual tool functions use; do not run untrusted notebook code after authorization.
- Colab free runtime availability is limited, and the Needle engine may download assets on first initialization.

In [ ]:
%pip install -q cactus-needle requests google-api-python-client

## 2. Import dependencies and check the runtime

In [ ]:
import json
import platform
import sys
import requests
import needle
from typing import Literal
from google.colab import auth
from googleapiclient.discovery import build

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Needle import: OK')

## 3. Authorize Google Drive

**User action required.** This cell opens Google's authorization flow. Read the prompt before approving. If you do not want this notebook to access your Drive, skip this cell and stop here.

In [ ]:
auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Google Drive API client initialized.')

## 4. Verify Drive API access with a metadata-only request

This lists at most five non-trashed file names and metadata. It does not download file contents.

In [ ]:
drive_test = drive_service.files().list(
    q='trashed = false',
    pageSize=5,
    orderBy='modifiedTime desc',
    fields='files(id,name,mimeType,modifiedTime,webViewLink)'
).execute()

print('Drive API: OK')
print('Metadata records returned:', len(drive_test.get('files', [])))
print(json.dumps([
    {k: f.get(k) for k in ('name', 'mimeType', 'modifiedTime', 'webViewLink')}
    for f in drive_test.get('files', [])
], indent=2, ensure_ascii=False))

## 5. Define the agent's read-only tools

The Drive search only accepts a filename substring and a small allow-list of file types. It does not accept raw Drive query syntax.

In [ ]:
@needle.tool(triggers=[r"(?i)\bgithub\b"])
def get_public_github_repo(owner: str, repo: str) -> dict:
    """Get metadata for a public GitHub repository using its owner and repository name."""
    allowed = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_."
    if (not owner or not repo or any(ch not in allowed for ch in owner + repo)
        or owner in {".", ".."} or repo in {".", ".."}):
        return {"ok": False, "error": "Invalid GitHub owner or repository name."}
    try:
        response = requests.get(
            f"https://api.github.com/repos/{owner}/{repo}",
            headers={"Accept": "application/vnd.github+json", "User-Agent": "AgentReach-Colab-Prototype"},
            timeout=15,
        )
        if response.status_code == 404:
            return {"ok": False, "error": "Public repository not found or not accessible."}
        response.raise_for_status()
        data = response.json()
        return {
            "ok": True,
            "name": data.get("full_name"),
            "description": data.get("description"),
            "url": data.get("html_url"),
            "language": data.get("language"),
            "stars": data.get("stargazers_count"),
            "default_branch": data.get("default_branch"),
        }
    except requests.RequestException as exc:
        return {"ok": False, "error": f"GitHub request failed: {type(exc).__name__}"}


@needle.tool(triggers=[r"(?i)\bgoogle drive\b", r"(?i)\bdrive files?\b"])
def search_google_drive_files(
    name_contains: str,
    file_type: Literal["any", "Google Docs", "Google Sheets", "Google Slides", "PDF"] = "any",
) -> dict:
    """Search Google Drive by a filename substring. Return up to 10 metadata-only matches. Allowed file_type values: any, Google Docs, Google Sheets, Google Slides, PDF."""
    if not isinstance(name_contains, str) or not name_contains.strip():
        return {"ok": False, "error": "Provide a non-empty filename substring."}
    name_contains = name_contains.strip()[:100]
    if any(ch in name_contains for ch in ("'", "\\", "\n", "\r")):
        return {"ok": False, "error": "For safety, the filename search cannot contain quote or backslash characters."}
    mime_types = {
        "Google Docs": "application/vnd.google-apps.document",
        "Google Sheets": "application/vnd.google-apps.spreadsheet",
        "Google Slides": "application/vnd.google-apps.presentation",
        "PDF": "application/pdf",
    }
    if not isinstance(file_type, str) or file_type not in ({"any"} | set(mime_types)):
        return {"ok": False, "error": "Unsupported file type."}
    query = f"name contains '{name_contains}' and trashed = false"
    if file_type != "any":
        query += f" and mimeType = '{mime_types[file_type]}'"
    try:
        response = drive_service.files().list(
            q=query,
            pageSize=10,
            orderBy="modifiedTime desc",
            fields="files(id,name,mimeType,modifiedTime,webViewLink)",
        ).execute()
        files = response.get("files", [])
        return {
            "ok": True,
            "count": len(files),
            "files": files,
            "note": "Metadata only; no file contents were read.",
        }
    except Exception as exc:
        return {"ok": False, "error": f"Drive lookup failed: {type(exc).__name__}"}


@needle.tool(triggers=[r"(?i)\bintegrations?\b", r"(?i)\bconnected apps\b"])
def list_supported_integrations() -> dict:
    """Report integrations available in this prototype and those not connected."""
    return {
        "available": [
            "Needle 3 local tool-calling runtime",
            "Public GitHub repository metadata (read-only, no GitHub token)",
            "Google Drive filename search (read-only metadata, after Google authorization)",
        ],
        "not_connected": [
            "Gmail",
            "Google Calendar",
            "Private GitHub access",
            "GitHub write actions",
            "Drive content reading or writing",
            "Browser automation",
            "Local file access",
        ],
    }


print("Defined three allow-listed tools.")


## 6. Initialize Needle 3

This can download the runtime/model asset on first use.

In [ ]:
agent = needle.Needle(tools=[get_public_github_repo, search_google_drive_files, list_supported_integrations])
print('Agent initialized. Available tools:', [
    get_public_github_repo.__name__, search_google_drive_files.__name__, list_supported_integrations.__name__
])

## 7. Test Google Drive filename search

Edit `drive_prompt` to a filename word you expect to find in your own Drive. The model should select the Drive tool.

In [ ]:
drive_prompt = "Search Google Drive for files with 'project' in the name. Use any file type."
drive_result = agent.run(drive_prompt)
print(json.dumps(drive_result, indent=2, ensure_ascii=False, default=str))
print('\nTool results:', json.dumps(drive_result.get('results', []), indent=2, ensure_ascii=False, default=str))

## 8. Test public GitHub lookup

In [ ]:
github_result = agent.run('Use GitHub to look up the public repository karpathy/nanochat.')
print(json.dumps(github_result, indent=2, ensure_ascii=False, default=str))

## 9. Ask your own tool-backed question

Change the prompt. Needle 3 is a tool-calling model, so requests outside the registered tools may correctly return no calls.

In [ ]:
user_prompt = "Which integrations are currently available, and which ones are not connected?"
custom_result = agent.run(user_prompt)
print(json.dumps(custom_result, indent=2, ensure_ascii=False, default=str))

## Success checklist
- Drive authorization completes and the metadata-only test runs.
- Google Drive search returns `ok: true`, even if zero files match the chosen name.
- Public GitHub lookup still works.
- No tool has write permissions in its implementation.

If a step fails, stop and share the full traceback with any personal file names redacted. Never share Google credentials, OAuth tokens, cookies, or authorization codes in this chat. The next phase can add reading a **specific file selected by ID** after we confirm this phase works; editing or deleting files will remain disabled until deliberate approval controls exist.